# 03 — Process Court documents

Convert downloaded Court HTML and PDF files into one UTF-8 plain-text file per source document. Preserve the raw files. HTML is decoded and cleaned lightly; PDFs use PyMuPDF, then `pdftotext`, then MinerU OCR when earlier extraction is technically suspicious. Usable imperfect text is kept with a `review` status.

This follows the same processing approach as the Commission notebooks; the resulting text can be used for later citation extraction.


## Inputs and outputs

Input: `output/court_infocuria/court_document_downloads.xlsx` from notebook 02.

Outputs: one file per source document in `data/processed/court_infocuria/text/` and `output/court_infocuria/court_clean_file_manifest.xlsx`, linking source files to text and recording extraction method and review status.

`logs/court_infocuria/processing_state.jsonl` lets interrupted processing resume.


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from bs4 import BeautifulSoup, UnicodeDammit
except Exception:
    BeautifulSoup = None
    UnicodeDammit = None

try:
    from charset_normalizer import from_bytes as charset_from_bytes
except Exception:
    charset_from_bytes = None


## 1. Paths and processing settings

Use the same repository structure as notebooks 01 and 02. Processing state and source hashes allow completed files to be reused safely after an interrupted run.


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "02_court").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
INPUT_MANIFEST = PROJECT_ROOT / "output" / "court_infocuria" / "court_document_downloads.xlsx"
PROCESSED_TEXT_DIR = PROJECT_ROOT / "data" / "processed" / "court_infocuria" / "text"
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
LOG_DIR = PROJECT_ROOT / "logs" / "court_infocuria"
CLEAN_FILE_MANIFEST_XLSX = OUTPUT_DIR / "court_clean_file_manifest.xlsx"
PROCESSING_STATE_JSONL = LOG_DIR / "processing_state.jsonl"

for directory in [PROCESSED_TEXT_DIR, OUTPUT_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FORCE_REPROCESS = False
RUN_PDFTOTEXT = True
RUN_OCR = True
PDFTOTEXT_TIMEOUT_SECONDS = 300
OCR_TIMEOUT_SECONDS = 900

MIN_TOTAL_CHARS = 200
MIN_PDF_CHARS_PER_PAGE = 500
MAX_BAD_ENCODING_MARKERS = 3
MAX_ONE_CHAR_TOKEN_RATIO = 0.20
BAD_ENCODING_MARKERS = ["�", "Ã", "Â", "â€™", "â€œ", "â€", "ðŸ"]
SUPPORTED_FILE_FORMATS = {"pdf", "html", "htm", "xhtml", "txt", "text"}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")


def repo_path(path: Path | str) -> str:
    path = Path(path).resolve()
    try:
        return path.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def resolve_repo_path(value: str) -> Path:
    value = str(value or "").strip()
    if not value:
        raise ValueError("Empty repository path")
    path = Path(value)
    if path.is_absolute():
        return path.resolve()
    candidate = (PROJECT_ROOT / path).resolve()
    if candidate.exists():
        return candidate
    # tolerate old rows that include the repository directory name
    if path.parts and path.parts[0] == PROJECT_ROOT.name:
        return (PROJECT_ROOT.parent / path).resolve()
    return candidate



def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("Project root:", PROJECT_ROOT)
print("Download manifest:", INPUT_MANIFEST)
print("Processed text:", PROCESSED_TEXT_DIR)
print("Clean-file manifest:", CLEAN_FILE_MANIFEST_XLSX)
print("Processing state:", PROCESSING_STATE_JSONL)


Project root: /home/edik/projects/ec_cjeu
Download manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_document_downloads.xlsx
Processed text: /home/edik/projects/ec_cjeu/data/processed/court_infocuria/text
Clean-file manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_clean_file_manifest.xlsx
Processing state: /home/edik/projects/ec_cjeu/logs/court_infocuria/processing_state.jsonl


## 2. Validate the download handoff

Only successfully downloaded local files enter text processing. The raw filename is preserved;
the canonical clean filename is obtained by replacing the source extension with `.txt`.


In [3]:
REQUIRED_COLUMNS = {
    "procedureId", "affId", "logicDocId", "normalized_docTypeCode",
    "raw_file_name", "raw_file_path", "file_format", "success",
}


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required input manifest is missing: {path}")
    return pd.read_excel(path, dtype=str, keep_default_na=False)


def as_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def first_value(row: pd.Series, columns: list[str]) -> str:
    for column in columns:
        if column in row.index:
            value = str(row.get(column, "") or "").strip()
            if value and value.lower() not in {"nan", "none", "<na>"}:
                return value
    return ""


download_manifest = read_manifest(INPUT_MANIFEST)
missing_columns = sorted(REQUIRED_COLUMNS - set(download_manifest.columns))
if missing_columns:
    raise ValueError(f"The download manifest is missing required columns: {missing_columns}")

successful = download_manifest.loc[
    as_bool(download_manifest["success"]) & download_manifest["raw_file_path"].ne("")
].copy()
if successful.empty:
    raise ValueError("No downloaded InfoCuria files are available for text processing.")

raw_paths = [resolve_repo_path(value) for value in successful["raw_file_path"]]
missing_files = [successful.iloc[i]["raw_file_name"] for i, path in enumerate(raw_paths) if not path.exists() or not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"{len(missing_files)} referenced raw files are missing; examples: {missing_files[:10]}")

successful["raw_path_internal"] = [str(path) for path in raw_paths]
successful["raw_file_name"] = [path.name for path in raw_paths]
successful["file_format_normalized"] = [
    (str(fmt).strip().lower().lstrip(".") or path.suffix.lower().lstrip("."))
    for fmt, path in zip(successful["file_format"], raw_paths)
]


def expected_text_name(raw_path: Path) -> str:
    source_type = raw_path.suffix.lower().lstrip(".")
    return f"{source_type}_{raw_path.stem}.txt"

successful["clean_file_name"] = [expected_text_name(path) for path in raw_paths]

# Repeated manifest rows may legitimately point to the same physical raw file.
# Those rows will share the same processed .txt file downstream.
# Only fail when DIFFERENT raw files would collide on the same clean filename.
clean_name_path_counts = (
    successful.groupby("clean_file_name")["raw_path_internal"]
    .nunique()
)
true_collisions = clean_name_path_counts.loc[lambda s: s > 1].index.tolist()
if true_collisions:
    raise ValueError(
        "Different raw files would create the same clean filename: "
        f"{true_collisions[:10]}"
    )

successful["record_id"] = [
    first_value(row, ["logicDocId", "docId", "idPublished", "document_dedup_key"])
    or f"infocuria_document_{i:06d}"
    for i, (_, row) in enumerate(successful.iterrows(), start=1)
]
if not successful["record_id"].is_unique:
    successful["record_id"] = [f"{rid}__{i:06d}" for i, rid in enumerate(successful["record_id"], start=1)]

work_manifest = successful.reset_index(drop=True)
print(f"Rows in download manifest: {len(download_manifest):,}")
print(f"Downloaded files available for processing: {len(work_manifest):,}")
display(work_manifest[[c for c in ["record_id", "case_number_clean", "normalized_docTypeCode", "raw_file_name", "file_format_normalized", "raw_file_path"] if c in work_manifest.columns]].head(20))


KeyboardInterrupt: 

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0xedeab261fdd0>>
Traceback (most recent call last):
  File "/home/edik/projects/.venv/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

## 3. Canonical text representation

Cleaning is deliberately light: Unicode and whitespace are normalized, line-ending hyphenation
is repaired, and readable paragraph structure is retained. The quality checks are used only to
trigger fallback extraction; they are not substantive legal-document quality scores.


In [ ]:
def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize(
        "NFKC",
        text or "",
    )

    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
        "\f": "\n\n",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def markdown_to_plain_text(text: str) -> str:
    """
    Conservative removal of common Markdown syntax from MinerU output.
    Legal wording is retained; only presentation markup is stripped.
    """
    text = text or ""

    # Images: keep alt text, discard target.
    text = re.sub(
        r"!\[([^\]]*)\]\([^)]+\)",
        r"\1",
        text,
    )

    # Links: keep visible text.
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text,
    )

    # Headings and block quotes.
    text = re.sub(
        r"(?m)^\s{0,3}#{1,6}\s+",
        "",
        text,
    )
    text = re.sub(
        r"(?m)^\s{0,3}>\s?",
        "",
        text,
    )

    # Fenced-code markers and horizontal rules.
    text = re.sub(
        r"(?m)^\s*```[A-Za-z0-9_-]*\s*$",
        "",
        text,
    )
    text = re.sub(
        r"(?m)^\s*(?:---+|\*\*\*+|___+)\s*$",
        "",
        text,
    )

    # Light inline emphasis cleanup.
    text = text.replace("**", "")
    text = text.replace("__", "")
    text = text.replace("`", "")

    return text


def clean_text(text: str) -> str:
    """Light normalization while preserving readable document structure."""
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")

    # Join words broken by line-ending hyphenation.
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n" if text.strip() else ""


def count_bad_encoding_markers(text: str) -> int:
    return sum(
        text.count(marker)
        for marker in BAD_ENCODING_MARKERS
    )


def one_char_token_ratio(text: str) -> float:
    tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        text or "",
    )

    if not tokens:
        return 1.0

    return (
        sum(len(token) == 1 for token in tokens)
        / len(tokens)
    )


## 4. Technical fallback checks

A small set of transparent checks identifies extractions that are too short or show strong signs of encoding or tokenization problems. These checks only trigger another extraction method; they are not substantive quality scores.


In [ ]:
def assess_text(
    text: str,
    *,
    file_format: str,
    n_pages: int | None,
) -> dict[str, Any]:
    text = text or ""

    n_chars = len(text)
    bad_marker_count = count_bad_encoding_markers(text)
    fragment_ratio = one_char_token_ratio(text)

    chars_per_page = None

    if file_format == "pdf" and n_pages:
        chars_per_page = n_chars / n_pages

    reasons: list[str] = []

    if n_chars < MIN_TOTAL_CHARS:
        reasons.append("too_little_text")

    if (
        file_format == "pdf"
        and chars_per_page is not None
        and chars_per_page < MIN_PDF_CHARS_PER_PAGE
    ):
        reasons.append("low_text_per_page")

    if bad_marker_count >= MAX_BAD_ENCODING_MARKERS:
        reasons.append("encoding_corruption")

    if fragment_ratio > MAX_ONE_CHAR_TOKEN_RATIO:
        reasons.append("fragmented_text")

    return {
        "passes": not reasons,
        "reasons": reasons,
        "n_chars": n_chars,
        "n_pages": n_pages,
        "chars_per_page": (
            round(chars_per_page, 1)
            if chars_per_page is not None
            else None
        ),
    }


## 5. HTML and plain-text extraction

HTML is decoded using declared encodings where available, followed by standard fallbacks.
Scripts, navigation, forms, cookie elements, and similar page chrome are removed before the
main text container is extracted.


In [ ]:
HTML_REMOVE_SELECTORS = [
    "script",
    "style",
    "noscript",
    "nav",
    "footer",
    "header",
    "form",
    "aside",
    "[class*='cookie']",
    "[id*='cookie']",
    "[class*='share']",
    "[class*='language']",
    "[aria-label*='language']",
    "[class*='pagination']",
    "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main",
    "article",
    "#document1",
    "#TexteOnly",
    "#text",
    ".document-content",
    ".document",
    ".content",
    "body",
]


def decode_issue_count(text: str) -> int:
    control_chars = sum(
        1
        for char in text
        if (
            unicodedata.category(char).startswith("C")
            and char not in "\n\t"
        )
    )

    return (
        count_bad_encoding_markers(text)
        + control_chars
    )


def extract_declared_html_encoding(
    data: bytes,
) -> str | None:
    head = data[:8192].decode(
        "ascii",
        errors="ignore",
    )

    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            head,
            flags=re.I,
        )

        if match:
            return match.group(1).strip()

    return None


def decode_bytes_smart(
    data: bytes,
    *,
    is_html: bool = False,
) -> tuple[str, dict[str, Any]]:
    candidates: list[tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(
            (
                "utf-8-sig",
                "bom",
                data.decode(
                    "utf-8-sig",
                    errors="replace",
                ),
            )
        )

    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(
            (
                "utf-16",
                "bom",
                data.decode(
                    "utf-16",
                    errors="replace",
                ),
            )
        )

    declared = (
        extract_declared_html_encoding(data)
        if is_html
        else None
    )

    if declared:
        try:
            candidates.append(
                (
                    declared,
                    "declared",
                    data.decode(
                        declared,
                        errors="replace",
                    ),
                )
            )
        except (LookupError, UnicodeError):
            pass

    try:
        candidates.append(
            (
                "utf-8",
                "strict_utf8",
                data.decode(
                    "utf-8",
                    errors="strict",
                ),
            )
        )
    except UnicodeDecodeError:
        pass

    if is_html and UnicodeDammit is not None:
        try:
            dammit = UnicodeDammit(
                data,
                is_html=True,
                smart_quotes_to=None,
            )

            if dammit.unicode_markup:
                candidates.append(
                    (
                        dammit.original_encoding or "unknown",
                        "unicode_dammit",
                        dammit.unicode_markup,
                    )
                )
        except Exception:
            pass

    if charset_from_bytes is not None:
        try:
            for match in list(
                charset_from_bytes(data)
            )[:3]:
                candidates.append(
                    (
                        getattr(
                            match,
                            "encoding",
                            None,
                        ) or "unknown",
                        "charset_normalizer",
                        str(match),
                    )
                )
        except Exception:
            pass

    for encoding in ("cp1252", "latin-1"):
        try:
            candidates.append(
                (
                    encoding,
                    "fallback",
                    data.decode(
                        encoding,
                        errors="replace",
                    ),
                )
            )
        except Exception:
            pass

    if not candidates:
        candidates.append(
            (
                "utf-8",
                "last_resort",
                data.decode(
                    "utf-8",
                    errors="replace",
                ),
            )
        )

    unique: list[tuple[str, str, str]] = []
    seen: set[str] = set()

    for encoding, source, text in candidates:
        digest = hashlib.sha1(
            text.encode(
                "utf-8",
                errors="ignore",
            )
        ).hexdigest()

        if digest not in seen:
            seen.add(digest)
            unique.append(
                (
                    encoding,
                    source,
                    text,
                )
            )

    encoding, source, text = min(
        unique,
        key=lambda item: decode_issue_count(
            item[2]
        ),
    )

    return normalize_unicode(text), {
        "encoding": encoding,
        "encoding_source": source,
    }


def extract_text_from_html(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    if BeautifulSoup is None:
        raise RuntimeError(
            "BeautifulSoup is not installed."
        )

    html, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=True,
    )

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    for selector in HTML_REMOVE_SELECTORS:
        try:
            for tag in soup.select(selector):
                tag.decompose()
        except Exception:
            pass

    candidates: list[tuple[int, str, str]] = []

    for selector in HTML_CANDIDATE_SELECTORS:
        try:
            for tag in soup.select(selector):
                text = tag.get_text(
                    "\n",
                    strip=True,
                )

                if len(text) >= 100:
                    candidates.append(
                        (
                            len(text),
                            selector,
                            text,
                        )
                    )
        except Exception:
            pass

    if candidates:
        _, selected_selector, text = max(
            candidates,
            key=lambda item: item[0],
        )
    else:
        selected_selector = "document"
        text = soup.get_text(
            "\n",
            strip=True,
        )

    return clean_text(text), {
        "method": "html",
        "n_pages": None,
        "html_selected_container": selected_selector,
        **decode_meta,
    }


def extract_text_from_plain(
    path: Path,
) -> tuple[str, dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(
        path.read_bytes(),
        is_html=False,
    )

    return clean_text(text), {
        "method": "plain_text",
        "n_pages": None,
        **decode_meta,
    }


## 6. PDF extraction

PDFs are first read with PyMuPDF. If the extracted text remains technically suspicious,
`pdftotext` is attempted, followed by MinerU OCR when necessary and available. MinerU may
produce Markdown internally, but presentation markup is removed before the canonical `.txt`
file is written.


In [ ]:
def extract_pdf_pymupdf(
    path: Path,
) -> tuple[str, int | None]:
    if fitz is None:
        raise RuntimeError(
            "PyMuPDF is not installed."
        )

    page_text: list[str] = []

    with fitz.open(path) as document:
        for page in document:
            page_text.append(
                page.get_text(
                    "text",
                    sort=True,
                ) or ""
            )

        n_pages = len(document)

    return (
        clean_text(
            "\n\n".join(page_text)
        ),
        n_pages,
    )


def extract_pdf_pdftotext(
    path: Path,
) -> tuple[str, int | None]:
    if PDFTOTEXT_CLI is None:
        raise RuntimeError(
            "pdftotext is not available on PATH."
        )

    with tempfile.TemporaryDirectory(
        prefix="pdftotext_"
    ) as tmpdir:
        output_path = (
            Path(tmpdir)
            / "output.txt"
        )

        command = [
            PDFTOTEXT_CLI,
            "-layout",
            "-enc",
            "UTF-8",
            str(path),
            str(output_path),
        ]

        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=PDFTOTEXT_TIMEOUT_SECONDS,
            check=False,
        )

        if completed.returncode != 0:
            raise RuntimeError(
                completed.stderr[-2000:]
                or "pdftotext failed"
            )

        text = output_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

    n_pages = None

    if fitz is not None:
        try:
            with fitz.open(path) as document:
                n_pages = len(document)
        except Exception:
            pass

    return clean_text(text), n_pages


def run_mineru(
    input_pdf: Path,
    output_dir: Path,
) -> None:
    if MINERU_CLI is None:
        raise RuntimeError(
            "MinerU is not available on PATH."
        )

    command = [
        MINERU_CLI,
        "-p",
        str(input_pdf),
        "-o",
        str(output_dir),
    ]

    environment = os.environ.copy()
    environment.setdefault(
        "ORT_LOG_SEVERITY_LEVEL",
        "3",
    )

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=OCR_TIMEOUT_SECONDS,
        env=environment,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            completed.stderr[-2000:]
            or "MinerU failed"
        )


def find_mineru_text(
    output_dir: Path,
) -> tuple[Path | None, bool]:
    """
    Return the largest non-empty MinerU text representation.

    The boolean indicates whether the selected intermediate file is Markdown.
    """
    markdown_files = [
        path
        for path in output_dir.glob("**/*.md")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if markdown_files:
        return (
            max(
                markdown_files,
                key=lambda path: path.stat().st_size,
            ),
            True,
        )

    text_files = [
        path
        for path in output_dir.glob("**/*.txt")
        if (
            path.is_file()
            and path.stat().st_size > 0
        )
    ]

    if text_files:
        return (
            max(
                text_files,
                key=lambda path: path.stat().st_size,
            ),
            False,
        )

    return None, False


def extract_pdf_mineru(
    path: Path,
) -> tuple[str, int | None]:
    with tempfile.TemporaryDirectory(
        prefix="mineru_"
    ) as tmpdir:
        output_dir = Path(tmpdir)

        run_mineru(
            path,
            output_dir,
        )

        extracted_path, is_markdown = find_mineru_text(
            output_dir
        )

        if extracted_path is None:
            raise RuntimeError(
                "MinerU completed but produced no non-empty "
                "text representation."
            )

        text = extracted_path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        if is_markdown:
            text = markdown_to_plain_text(text)

    n_pages = None

    if fitz is not None:
        try:
            with fitz.open(path) as document:
                n_pages = len(document)
        except Exception:
            pass

    return clean_text(text), n_pages


def try_pdf_extractor(
    path: Path,
    method: str,
    extractor,
) -> tuple[dict[str, Any] | None, str]:
    try:
        text, n_pages = extractor(path)

        assessment = assess_text(
            text,
            file_format="pdf",
            n_pages=n_pages,
        )

        return {
            "method": method,
            "text": text,
            "assessment": assessment,
        }, ""

    except Exception as exc:
        return None, repr(exc)


## 7. Deterministic fallback and review status

The first extraction that passes the technical checks is retained. If every available method
remains suspicious but yields non-empty text, the best available text is saved with
`processing_status = review`. `failed` is reserved for files from which no usable text can be
obtained.


In [ ]:
def process_pdf(
    path: Path,
) -> dict[str, Any]:
    attempts: list[dict[str, Any]] = []
    errors: list[str] = []
    fallback_reason: list[str] = []

    # 1. Default native extraction: PyMuPDF
    pymupdf_result, error = try_pdf_extractor(
        path,
        "pymupdf",
        extract_pdf_pymupdf,
    )

    if error:
        errors.append(
            f"pymupdf: {error}"
        )
        fallback_reason.append(
            "pymupdf_failed"
        )

    elif pymupdf_result is not None:
        attempts.append(
            pymupdf_result
        )

        if pymupdf_result[
            "assessment"
        ]["passes"]:
            return {
                "selected": pymupdf_result,
                "status": "success",
                "review_reason": "",
                "ocr_used": False,
                "fallback_reason": "",
                "errors": errors,
            }

        fallback_reason.extend(
            f"pymupdf:{reason}"
            for reason in pymupdf_result[
                "assessment"
            ]["reasons"]
        )

    # 2. Native fallback: pdftotext -layout
    if (
        RUN_PDFTOTEXT
        and PDFTOTEXT_CLI is not None
    ):
        poppler_result, error = try_pdf_extractor(
            path,
            "pdftotext",
            extract_pdf_pdftotext,
        )

        if error:
            errors.append(
                f"pdftotext: {error}"
            )
            fallback_reason.append(
                "pdftotext_failed"
            )

        elif poppler_result is not None:
            attempts.append(
                poppler_result
            )

            if poppler_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": poppler_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": False,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            fallback_reason.extend(
                f"pdftotext:{reason}"
                for reason in poppler_result[
                    "assessment"
                ]["reasons"]
            )

    elif RUN_PDFTOTEXT:
        fallback_reason.append(
            "pdftotext_unavailable"
        )

    # 3. OCR fallback: MinerU
    if (
        RUN_OCR
        and MINERU_CLI is not None
    ):
        ocr_result, error = try_pdf_extractor(
            path,
            "mineru_ocr",
            extract_pdf_mineru,
        )

        if error:
            errors.append(
                f"mineru_ocr: {error}"
            )
            fallback_reason.append(
                "mineru_failed"
            )

        elif (
            ocr_result is not None
            and ocr_result["text"].strip()
        ):
            attempts.append(
                ocr_result
            )

            if ocr_result[
                "assessment"
            ]["passes"]:
                return {
                    "selected": ocr_result,
                    "status": "success",
                    "review_reason": "",
                    "ocr_used": True,
                    "fallback_reason": "; ".join(
                        fallback_reason
                    ),
                    "errors": errors,
                }

            return {
                "selected": ocr_result,
                "status": "review",
                "review_reason": "; ".join(
                    f"mineru:{reason}"
                    for reason in ocr_result[
                        "assessment"
                    ]["reasons"]
                ),
                "ocr_used": True,
                "fallback_reason": "; ".join(
                    fallback_reason
                ),
                "errors": errors,
            }

    elif RUN_OCR:
        fallback_reason.append(
            "mineru_unavailable"
        )

    # If nothing passed, keep the latest non-empty native extraction.
    nonempty_attempts = [
        attempt
        for attempt in attempts
        if attempt.get(
            "text",
            "",
        ).strip()
    ]

    if nonempty_attempts:
        selected = nonempty_attempts[-1]

        return {
            "selected": selected,
            "status": "review",
            "review_reason": "; ".join(
                f"{selected['method']}:{reason}"
                for reason in selected[
                    "assessment"
                ]["reasons"]
            ) or "no_method_passed_checks",
            "ocr_used": (
                selected["method"]
                == "mineru_ocr"
            ),
            "fallback_reason": "; ".join(
                fallback_reason
            ),
            "errors": errors,
        }

    return {
        "selected": None,
        "status": "failed",
        "review_reason": "no_text_extracted",
        "ocr_used": False,
        "fallback_reason": "; ".join(
            fallback_reason
        ),
        "errors": errors,
    }


def process_non_pdf(
    path: Path,
    file_format: str,
) -> dict[str, Any]:
    extractor = (
        extract_text_from_html
        if file_format in {
            "html",
            "htm",
            "xhtml",
            "xml",
        }
        else extract_text_from_plain
    )

    try:
        text, meta = extractor(path)

        assessment = assess_text(
            text,
            file_format=file_format,
            n_pages=None,
        )

    except Exception as exc:
        return {
            "selected": None,
            "status": "failed",
            "review_reason": "extraction_failed",
            "ocr_used": False,
            "fallback_reason": "",
            "errors": [repr(exc)],
        }

    selected = {
        "method": meta["method"],
        "text": text,
        "assessment": assessment,
    }

    if not text.strip():
        status = "failed"
        review_reason = "no_text_extracted"

    elif assessment["passes"]:
        status = "success"
        review_reason = ""

    else:
        status = "review"
        review_reason = "; ".join(
            assessment["reasons"]
        )

    return {
        "selected": (
            selected
            if text.strip()
            else None
        ),
        "status": status,
        "review_reason": review_reason,
        "ocr_used": False,
        "fallback_reason": "",
        "errors": [],
    }


## 8. Resumable processing state

The JSONL checkpoint records the latest processing result for each document. A previous result
is reused only when the raw file hash, clean-file path, and clean-file hash still agree.


In [ ]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (str, int, float, bool),
    ):
        if (
            isinstance(value, float)
            and pd.isna(value)
        ):
            return None

        return value

    if isinstance(value, Path):
        return value.as_posix()

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return str(value)


def load_processing_state() -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}

    if not PROCESSING_STATE_JSONL.exists():
        return latest

    with PROCESSING_STATE_JSONL.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # Ignore a malformed partial final line after interruption.
                continue

            record_id = str(
                record.get(
                    "record_id",
                    "",
                )
            ).strip()

            if record_id:
                latest[record_id] = record

    return latest


def append_processing_state(
    record: dict[str, Any],
) -> None:
    with PROCESSING_STATE_JSONL.open(
        "a",
        encoding="utf-8",
    ) as handle:
        handle.write(
            json.dumps(
                json_safe(record),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


def expected_text_path(
    raw_path: Path,
) -> Path:
    return (
        PROCESSED_TEXT_DIR
        / expected_text_name(raw_path)
    )


def state_is_reusable(
    state_record: dict[str, Any] | None,
    *,
    raw_sha256: str,
    text_path: Path,
) -> bool:
    if FORCE_REPROCESS or not state_record:
        return False

    if state_record.get(
        "processing_status"
    ) not in {"success", "review"}:
        return False
    if (
        str(
            state_record.get(
                "raw_sha256",
                "",
            )
        ).lower()
        != raw_sha256.lower()
    ):
        return False

    if (
        state_record.get("clean_file_path")
        != repo_path(text_path)
    ):
        return False

    if (
        not text_path.exists()
        or text_path.stat().st_size <= 0
    ):
        return False

    expected_text_sha = str(
        state_record.get(
            "clean_sha256",
            "",
        )
    ).lower()

    if not re.fullmatch(
        r"[0-9a-f]{64}",
        expected_text_sha,
    ):
        return False

    return (
        sha256_file(text_path)
        == expected_text_sha
    )


def process_file(
    input_row: dict[str, Any],
    *,
    raw_sha256: str,
) -> dict[str, Any]:
    record_id = str(
        input_row["record_id"]
    )

    raw_path = Path(
        input_row["raw_path_internal"]
    )

    file_format = str(
        input_row["file_format_normalized"]
    ).strip().lower()

    text_path = expected_text_path(
        raw_path
    )

    base = {
        "timestamp_utc": utc_now_iso(),
        "record_id": record_id,
        "raw_file_name": raw_path.name,
        "raw_file_path": repo_path(raw_path),
        "raw_sha256": raw_sha256,
    }

    if file_format not in SUPPORTED_FILE_FORMATS:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": "",
            "review_reason": "unsupported_file_format",
            "clean_file_name": "",
            "clean_file_path": "",
            "clean_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                "unsupported_file_format:"
                f"{file_format}"
            ),
        }

    result = (
        process_pdf(raw_path)
        if file_format == "pdf"
        else process_non_pdf(
            raw_path,
            file_format,
        )
    )

    selected = result["selected"]

    if selected is None:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": result.get(
                "fallback_reason",
                "",
            ),
            "review_reason": result.get(
                "review_reason",
                "no_text_extracted",
            ),
            "clean_file_name": "",
            "clean_file_path": "",
            "clean_sha256": "",
            "n_pages": None,
            "n_chars": 0,
            "chars_per_page": None,
            "error": (
                " | ".join(
                    result.get(
                        "errors",
                        [],
                    )
                )
                or "no_text_extracted"
            ),
        }

    text = selected["text"]

    text_path.write_text(
        text,
        encoding="utf-8",
    )

    assessment = selected[
        "assessment"
    ]

    return {
        **base,
        "processing_status": result["status"],
        "extraction_method": selected["method"],
        "ocr_used": bool(
            result["ocr_used"]
        ),
        "fallback_reason": result.get(
            "fallback_reason",
            "",
        ),
        "review_reason": result.get(
            "review_reason",
            "",
        ),
        "clean_file_name": text_path.name,
        "clean_file_path": repo_path(text_path),
        "clean_sha256": sha256_file(text_path),
        "n_pages": assessment.get("n_pages"),
        "n_chars": assessment.get("n_chars", len(text)),
        "chars_per_page": assessment.get("chars_per_page"),
        "error": " | ".join(
            result.get(
                "errors",
                [],
            )
        ),
    }


## 9. Process the Court documents


In [ ]:
processing_state = load_processing_state()
manifest_rows: list[dict[str, Any]] = []
run_counts = {"success": 0, "review": 0, "failed": 0, "ocr": 0, "reused": 0}

# Cache results by physical raw-file path so duplicate manifest rows
# reuse the already extracted text instead of processing the file again.
raw_file_cache: dict[str, dict[str, Any]] = {}

for input_row in tqdm(work_manifest.to_dict("records"), desc="Processing InfoCuria documents", unit="file", dynamic_ncols=True):
    record_id = str(input_row["record_id"])
    raw_path = Path(input_row["raw_path_internal"])
    actual_raw_sha256 = sha256_file(raw_path)
    text_path = expected_text_path(raw_path)
    previous_state = processing_state.get(record_id)
    raw_cache_key = str(raw_path.resolve())

    if raw_cache_key in raw_file_cache:
        # The same physical file already appeared in an earlier manifest row.
        # Keep this row, but point it to the already-created clean text file.
        cached = raw_file_cache[raw_cache_key]
        state_record = {
            **cached,
            "timestamp_utc": utc_now_iso(),
            "record_id": record_id,
        }
        append_processing_state(state_record)
        processing_state[record_id] = state_record
        run_counts["reused"] += 1
    elif (not FORCE_REPROCESS) and state_is_reusable(
        previous_state,
        raw_sha256=actual_raw_sha256,
        text_path=text_path,
    ):
        state_record = previous_state
        raw_file_cache[raw_cache_key] = state_record
        run_counts["reused"] += 1
    else:
        state_record = process_file(input_row, raw_sha256=actual_raw_sha256)
        append_processing_state(state_record)
        processing_state[record_id] = state_record
        raw_file_cache[raw_cache_key] = state_record

    status = state_record.get("processing_status", "failed")
    run_counts[status if status in run_counts else "failed"] += 1
    if state_record.get("ocr_used"):
        run_counts["ocr"] += 1

    manifest_rows.append({
        "record_id": record_id,
        "procedureId": input_row.get("procedureId", ""),
        "affId": input_row.get("affId", ""),
        "case_number_clean": input_row.get("case_number_clean", ""),
        "logicDocId": input_row.get("logicDocId", ""),
        "document_type": input_row.get("normalized_docTypeCode", ""),
        "document_date": input_row.get("docDate", ""),
        "source_url": input_row.get("working_url", "") or input_row.get("download_url", ""),
        "raw_file_name": raw_path.name,
        "raw_file_path": repo_path(raw_path),
        "clean_file_name": state_record.get("clean_file_name", ""),
        "clean_file_path": state_record.get("clean_file_path", ""),
        "file_format": input_row.get("file_format_normalized", ""),
        "extraction_method": state_record.get("extraction_method", ""),
        "ocr_used": state_record.get("ocr_used", False),
        "processing_status": status,
        "review_reason": state_record.get("review_reason", ""),
        "fallback_reason": state_record.get("fallback_reason", ""),
        "raw_sha256": actual_raw_sha256,
        "clean_sha256": state_record.get("clean_sha256", ""),
        "n_pages": state_record.get("n_pages"),
        "n_chars": state_record.get("n_chars", 0),
        "error": state_record.get("error", ""),
    })

processing_manifest = pd.DataFrame(manifest_rows).reset_index(drop=True)
print(f"Files considered: {len(processing_manifest):,}")
print(f"Successful: {run_counts['success']:,}")
print(f"Review: {run_counts['review']:,}")
print(f"Failed: {run_counts['failed']:,}")
print(f"OCR selected: {run_counts['ocr']:,}")
print(f"Checkpoints reused: {run_counts['reused']:,}")


## 10. Save the clean-file manifest

The clean-file manifest is the publication-oriented handoff from raw acquisition to processed
text. It contains the raw and clean filenames and paths together with the extraction method,
processing status, hashes, and a compact review reason when needed.


In [ ]:
processing_manifest.to_excel(CLEAN_FILE_MANIFEST_XLSX, index=False, sheet_name="clean_files")

print("Saved:", repo_path(CLEAN_FILE_MANIFEST_XLSX))
display(processing_manifest[[
    "record_id", "case_number_clean", "document_type",
    "raw_file_name", "clean_file_name", "extraction_method", "ocr_used", "processing_status"
]].head(20))

print("\nProcessing status:")
display(processing_manifest["processing_status"].value_counts(dropna=False).rename_axis("processing_status").reset_index(name="files"))

review_rows = processing_manifest.loc[processing_manifest["processing_status"].isin(["review", "failed"])]
print(f"Rows requiring attention: {len(review_rows):,}")
if len(review_rows):
    display(review_rows[["record_id", "raw_file_name", "clean_file_name", "extraction_method", "processing_status", "review_reason", "error"]].head(30))


## 11. Reproducibility notes

Raw source files are preserved unchanged. Clean text is UTF-8 and deterministic for a fixed
set of extraction tools and settings. SHA-256 values in the clean-file manifest identify the
exact raw and processed files used in downstream analysis.
